# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Вы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузите данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. Если работаете локально, скачать файл можно по ссылке.

Проверьте наличие дубликатов в идентификаторах пользователей. Сравните размеры групп, их статистики и распределение.

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовьте аналитическую записку, в которой опишите:

* Выбранный тип t-теста и уровень статистической значимости.

* Результат теста, или p-value.

* Вывод на основе полученного p-value, то есть интерпретацию результатов.

* Одну или две возможные причины, объясняющие полученные результаты.

<h2>1. Напишите заголовок первой части проекта здесь</h2>

- Автор:Сорокина Мария
- Дата: 21.12.2025

<h2>2. Цели и задачи проекта</h2>

<font color='#777778'>В этом блоке перечислите цель проекта и те задачи, которые вы решаете. Можно использовать описания проекта, но будет полезно, если вы сформулируете основную цель проекта самостоятельно.</font>

1. Проанализировать данные о поведении пользователей сервиса Яндекс книги из Санкт-Петербурга и Москвы.
2. Провести t-тест и проверить составленные гипотезы.
3. Сделать вывод о реузльтатах теста.

<h2>Описание таблиц</h2>

<h3>Таблица yandex_knigi_data</h3>
<p>Содержит данные об активности пользователей и состоит из следующих полей:</p>
<ul>
    <li>city — название города;</li>
    <li>puid — идентификатор пользователя;</li>
    <li>hours — общее количество прослушанных часов;</li>
</ul>

<h3>Таблица ab_test_participants</h3>
<p>Содержит данные о проведении A/B теста и состоит из следующих полей:</p>
<ul>
    <li>user_id — идентификатор пользователя;</li>
    <li>group — группа пользователя;</li>
    <li>ab_test — название теста('interface_eu_test' 'recommender_system_test')</li>
    <li>device — устройство, с которого происходила регистрация.</li>
</ul>

<h3>Таблица ab_test_events</h3>
<p>Содержит архив с одним csv-файлом, в котором собраны события 2020 года и состоит из следующих полей:</p>
<ul>
    <li>user_id — идентификатор пользователя;</li>
    <li>event_dt — дата и время события;</li>
    <li>event_name — тип события;</li>
    <li>details — дополнительные данные о событии.</li>
</ul>

<div class="alert alert-info">
<h2>Содержимое проекта</h2>
В рамках проекта вам предстоит:
<ol>
<li>Провести предобработку данных. Проверить наличие дубликатов в идентификаторах пользователей. Сравнить размеры групп, их статистики и распределение.</li>
<li>Провести тест по сформулированной гипотезе и отобразить результаты теста  что это означает для исследователей.</li>
<li>По результатам анализа данных подготовьте аналитическую записку с описанием проведенных исследований и интерперетацией результатов.</li>
    </ol>
</div

## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
# Пустые ячейки после каждого задания — примерное пространство для работы.
# Вы можете свободно добавлять или удалять ячейки по своему усмотрению в зависимости от логики и объёма работы.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats as st
import math

knigi = pd.read_csv('https://code.s3.yandex.net/datasets/yandex_knigi_data.csv')

In [2]:
def check_df(dataset_name):
    print('Столбцы:')
    print(list(dataset_name.columns))
    print('-----------------------')
    print('Информация о датасете:')
    print(dataset_name.info())
    
check_df(knigi)

Столбцы:
['Unnamed: 0', 'city', 'puid', 'hours']
-----------------------
Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB
None


In [3]:
knigi.head(5)

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


In [4]:
for i in knigi.columns:
    print('-----------------')
    print(f'Уникальные значения столбца: {i}')
    print(knigi[i].unique())

-----------------
Уникальные значения столбца: Unnamed: 0
[   0    1    2 ... 8781 8782 8783]
-----------------
Уникальные значения столбца: city
['Москва' 'Санкт-Петербург']
-----------------
Уникальные значения столбца: puid
[            9668            16598            80401 ... 1130000038726322
 1130000047892100 1130000061443598]
-----------------
Уникальные значения столбца: hours
[26.16777639 82.11121667  4.65690606 ...  0.21194444  4.31184146
 20.84722222]


In [5]:
def nan_fraction(df):
    missing_count = df.isna().sum()
    total_rows = df.shape[0]
    missing_proportion = missing_count / total_rows
    print(missing_proportion)
nan_fraction(knigi)

Unnamed: 0    0.0
city          0.0
puid          0.0
hours         0.0
dtype: float64


In [6]:
knigi.drop_duplicates(inplace=True)
print(knigi.shape)

(8784, 4)


In [7]:
count_duplicates = knigi['puid'].duplicated()
print(f'Количество дубликатов среди пользователей:{count_duplicates.sum()}')
duplicates = knigi[knigi['puid'].duplicated(keep=False)]
# Вытаскиваем уникальные ID пользователей с дубликатами
duplicate_user_ids = duplicates['puid'].unique()
knigi[knigi['puid'].isin(duplicate_user_ids)].sort_values('puid')

Количество дубликатов среди пользователей:244


,Unnamed: 0,city,puid,hours
35,35,Москва,2637041,10.317371
6247,6247,Санкт-Петербург,2637041,3.883926
134,134,Москва,9979490,32.415573
6274,6274,Санкт-Петербург,9979490,1.302997
145,145,Москва,10597984,42.931506
...,...,...,...,...
6195,6195,Москва,1130000020425037,0.310556
8775,8775,Санкт-Петербург,1130000023864516,14.384722
6202,6202,Москва,1130000023864516,142.830085
6210,6210,Москва,1130000028554332,11.277554


<div class="alert alert-block alert-info">
Есть 244 пользователя, которые имеют данные для обоих городов, что может нарушить независимость групп для теста. Удалим их из выборки:
</div>

In [8]:
filtered_knigi = knigi[~knigi['puid'].isin(duplicate_user_ids)]
print(filtered_knigi['puid'].duplicated().sum())

0


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
group_msk = filtered_knigi[filtered_knigi['city'] == 'Москва']['hours']
group_spb = filtered_knigi[filtered_knigi['city'] == 'Санкт-Петербург']['hours']

# Уровень статистической значимости: можно указать .05 или 0.05
alpha = 0.05 

results = st.ttest_ind(
    group_msk, 
    group_spb,
    alternative='less' # Альтернативная гипотеза, которую проверяем: mu1 < mu2
)

print('p-значение:', results.pvalue)

if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу')
else:
    print('Нулевая гипотеза подтвердилась')

p-значение: 0.3264137070357819
Нулевая гипотеза подтвердилась


<div class="alert alert-block alert-info">
P значение двустороннего теста равно  0.326. Это значит, что нулевая гипотеза подтвердилась и средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.
        
</div>

In [10]:
mean_msk = filtered_knigi[filtered_knigi['city'] == 'Москва']['hours'].mean()
print(f'Средняя активность слушателей в Москве: {mean_msk}')
mean_spb = filtered_knigi[filtered_knigi['city'] == 'Санкт-Петербург']['hours'].mean()
print(f'Средняя активность слушателей в Санкт-Петербурге: {mean_spb}')

Средняя активность слушателей в Москве: 10.84819150713916
Средняя активность слушателей в Санкт-Петербурге: 11.264433367029522


## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



<div class="alert alert-block alert-info">
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:
<ul>
<li>Выбранный тип t-теста и уровень статистической значимости:
    t-test для двух выборок</li>  

<li>Результат теста, или p-value = 0.326</li>

<li>Вывод на основе полученного p-value, то есть интерпретацию результатов:
Это значит, что нулевая гипотеза подтвердилась и средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.</li>

<li>Одну или две возможные причины, объясняющие полученные результаты:
    <ol>
<li>Подобное поведение пользователей: пользователи в обоих городах могут вести себя в приложении одинаково, что приводит к схожим временным показателям. Это может быть связано с тем, что целевая аудитория и их потребности в обоих городах очень похожи.</li>

        

</ul>
</div>

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



1. Проанализировать предоставленные данные, при необходимости обработать.
2. Ознакомиться с техническим заданием проведенного теста от заказчика.
3. Оценить корректность проведения теста
4. Сделать вывод о результах теста.

## 2. Загрузите данные, оцените их целостность.


In [11]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

<h1>Техническое задание проведённого теста</h1>
Предыдущий аналитик проверял полное обновление дизайна сайта. <br>
Гипотеза заключается в следующем:<br>
<b>упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации в системе конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта.</b><br>
Параметры теста:
<ul>
<li>название теста: interface_eu_test;</li>
<li>группы: А (контрольная), B (новый интерфейс).</li>
    </ul>
Вам нужно:<br>
<ul>
<li>загрузить данные теста;</li>
<li>проверить корректность его проведения;</li>
<li>проанализировать полученные результаты.</li>
    </ul>
<b>Данные</b><br>
https://code.s3.yandex.net/datasets/ab_test_participants.csv — таблица участников тестов.<br>
Структура файла:
<ul>
<li>user_id — идентификатор пользователя;</li>
<li>group — группа пользователя;</li>
<li>ab_test — название теста;</li>
<li>device — устройство, с которого происходила регистрация.</li></ul>
https://code.s3.yandex.net/datasets/ab_test_events.zip — архив с одним csv-файлом, в котором собраны события 2020 года;<br>
Структура файла:<ul>
<li>user_id — идентификатор пользователя;</li>
<li>event_dt — дата и время события;</li>
<li>event_name — тип события;</li>
<li>details — дополнительные данные о событии.</li></ul>

## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [12]:
check_df(participants)

Столбцы:
['user_id', 'group', 'ab_test', 'device']
-----------------------
Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB
None


In [13]:
participants.head(5)

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [14]:
for i in participants.columns:
    print('-----------------')
    print(f'Уникальные значения столбца: {i}')
    print(participants[i].unique())

-----------------
Уникальные значения столбца: user_id
['0002CE61FF2C4011' '001064FEAAB631A1' '0010A1C096941592' ...
 'FFEFC0E55C1CCD4F' 'FFF28D02B1EACBE1' 'FFF58BC33966EB51']
-----------------
Уникальные значения столбца: group
['B' 'A']
-----------------
Уникальные значения столбца: ab_test
['interface_eu_test' 'recommender_system_test']
-----------------
Уникальные значения столбца: device
['Mac' 'Android' 'iPhone' 'PC']


In [15]:
nan_fraction(participants)

user_id    0.0
group      0.0
ab_test    0.0
device     0.0
dtype: float64


In [16]:
print(participants.shape)
participants.drop_duplicates(inplace=True)
print(participants.shape)

(14525, 4)
(14525, 4)


In [17]:
participants['ab_test_group'] = participants['ab_test'] + '_' + participants['group']

user_counts = participants.groupby('user_id')['ab_test_group'].nunique()

filtered_participants = participants[participants['user_id'].isin(user_counts[user_counts == 1].index)]

filtered_participants = filtered_participants[filtered_participants['ab_test'] == 'interface_eu_test']

filtered_participants['group'].value_counts()


B    5011
A    4952
Name: group, dtype: int64

3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [18]:
check_df(events)

Столбцы:
['user_id', 'event_dt', 'event_name', 'details']
-----------------------
Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB
None


In [19]:
events.head(5)

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [20]:
for i in events.columns:
    print('-----------------')
    print(f'Уникальные значения столбца: {i}')
    print(events[i].unique())

-----------------
Уникальные значения столбца: user_id
['GLOBAL' 'CCBE9E7E99F94A08' 'AA346F4D22148024' ... 'B77B2F4BCA134618'
 'B12AD1623E494FAD' '5456977474344433']
-----------------
Уникальные значения столбца: event_dt
['2020-12-01T00:00:00.000000000' '2020-12-01T00:00:11.000000000'
 '2020-12-01T00:00:25.000000000' ... '2020-12-31T23:58:30.000000000'
 '2020-12-31T23:58:34.000000000' '2020-12-31T23:59:48.000000000']
-----------------
Уникальные значения столбца: event_name
['End of Black Friday Ads Campaign' 'registration' 'product_page' 'login'
 'product_cart' 'purchase' 'Start of Christmas&New Year Promo'
 'Start of CIS New Year Gift Lottery']
-----------------
Уникальные значения столбца: details
['ZONE_CODE15' '0.0' nan '-2.0' '-5.0' '-3.5' '-1.5' '-0.5' '-2.5' '-4.0'
 '-4.5' '4.99' '9.99' '-6.0' '-5.5' '-3.0' '-1.0' '499.99' '-7.0' '99.99'
 '-6.5' '4.59' '4.890000000000001' '-7.5' '-9.0' '-8.5' '-9.5' '-8.0'
 '9.49' '9.59' '99.89' '99.58999999999999' '4.49' '9.89' '-0.48' '-2.38

In [21]:
nan_fraction(events)

user_id       0.000000
event_dt      0.000000
event_name    0.000000
details       0.683696
dtype: float64


<div class="alert alert-block alert-info">
        столбец details нам не важен если имеет пропуски
</div>

In [22]:
print(events.shape)
events.drop_duplicates(inplace=True)
print(events.shape)

(787286, 4)
(787286, 4)


In [23]:
inter_part_users = list(filtered_participants['user_id'])

In [24]:
events_int = events[events['user_id'].isin(inter_part_users)]

- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [25]:
min_event_dates = events_int.groupby('user_id')['event_dt'].min().reset_index()
min_event_dates.columns = ['user_id', 'min_event_dt']

# Присоединяем минимальные даты событий к исходному датафрейму
events_int = events_int.merge(min_event_dates, on='user_id', how='left')

# Рассчитываем разницу между датой события и минимальной датой события пользователя
events_int['time_since_min_event'] = events_int['event_dt'] - events_int['min_event_dt']

# Устанавливаем порог в 7 дней
threshold = pd.Timedelta(days=7)

# Отфильтровываем события, оставив только те, что произошли в течение первых семи дней
filtered_events_int = events_int[events_int['time_since_min_event'] <= threshold]

filtered_events_int

,user_id,event_dt,event_name,details,min_event_dt,time_since_min_event
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0 days 00:00:00
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0 days 00:00:00
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0 days 00:00:00
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0 days 00:00:00
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0 days 00:00:00
...,...,...,...,...,...,...
73667,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN,2020-12-23 09:35:48,6 days 12:10:55
73670,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN,2020-12-23 09:35:48,6 days 12:12:08
73739,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN,2020-12-23 13:53:33,6 days 08:53:27
73745,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN,2020-12-23 13:53:33,6 days 08:55:13


In [26]:
group_a_inter_part = list(filtered_participants[filtered_participants['group'] == 'A']['user_id'])
group_b_inter_part = list(filtered_participants[filtered_participants['group'] == 'B']['user_id'])


filtered_events_int_a = filtered_events_int[filtered_events_int['user_id'].isin(group_a_inter_part)]
filtered_events_int_b = filtered_events_int[filtered_events_int['user_id'].isin(group_b_inter_part)]

Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [27]:
alpha = 0.05  # Уровень значимости
beta = 0.2  # Ошибка второго рода, часто 1 - мощность
power = 0.8  # Мощность теста
p1 = 0.3 # Базовый уровень доли
mde = 0.1 * p1  # Минимальный детектируемый эффект
effect_size = proportion_effectsize(p1, p1 + mde)

power_analysis = NormalIndPower()

sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1 # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


In [28]:
print(filtered_events_int_a.shape[0]/sample_size)
print(filtered_events_int_b.shape[0]/sample_size)

8.311363555424663
8.6508470271582


<div class="alert alert-block alert-info">
        наблюдений достаточно для параметров теста
</div>

- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [29]:
group_a_purch_users = filtered_events_int_a[filtered_events_int_a['event_name'] == 'purchase']['user_id'].unique()
print(group_a_purch_users.shape[0])
group_a_all_users = filtered_events_int_a['user_id'].unique()
print(group_a_all_users.shape[0])

group_b_purch_users = filtered_events_int_b[filtered_events_int_b['event_name'] == 'purchase']['user_id'].unique()
print(group_b_purch_users.shape[0])
group_b_all_users = filtered_events_int_b['user_id'].unique()
print(group_b_all_users.shape[0])

1377
4952
1480
5011


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

In [30]:
print(group_a_purch_users.shape[0]/group_a_all_users.shape[0])
print(group_b_purch_users.shape[0]/group_b_all_users.shape[0])
print(round(group_b_purch_users.shape[0]/group_b_all_users.shape[0] - group_a_purch_users.shape[0]/group_a_all_users.shape[0],2))

0.27806946688206785
0.29535022949511075
0.02


<div class="alert alert-block alert-info">
        Пользовательская активность в тестовой группе выше, чем в контрольной примерно на 2%.
</div>

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [31]:
filtered_events_int_a_conv = filtered_events_int_a.groupby('user_id').agg(
    purchase_flag=('event_name', lambda x: any(x == 'purchase'))
).reset_index()
filtered_events_int_b_conv = filtered_events_int_b.groupby('user_id').agg(
    purchase_flag=('event_name', lambda x: any(x == 'purchase'))
).reset_index()

<div class="alert alert-block alert-info">
        Гипотезы:
        Нулевая гипотеза: вероятность наступления события регистрация и далее покупка у двух групп одинаковая.<br>
        Альтернативная гипотеза: вероятность наступления события регистрация и далее покупка у тестовой выборки выше.
</div>

In [32]:
print(f'n_a={len(group_a_all_users)}, n_b={len(group_b_all_users)}')

print(f'm_a={len(group_a_purch_users)}, m_b={len(group_b_purch_users)}')

p_a, p_b = len(group_a_purch_users)/len(group_a_all_users), len(group_b_purch_users)/len(group_b_all_users)


print(f'p_a={p_a}, p_b={p_b}')

if (p_a*len(group_a_all_users) > 10)and((1-p_a)*len(group_a_all_users) > 10)and(p_b*len(group_b_all_users) > 10)and((1-p_b)*len(group_b_all_users) > 10):
    print('Предпосылка о достаточном количестве данных выполняется!')
else:
    print('Предпосылка о достаточном количестве данных НЕ выполняется!')
n_a, n_b = len(group_a_all_users), len(group_b_all_users)# берём предыдущие значения по размерам выборок A и B 
m_a, m_b = len(group_a_purch_users), len(group_b_purch_users) # берём предыдущие значения по количеству успехов в выборках A и B 

alpha = 0.05 ## на каком уровне значимости проверяем гипотезу о равенстве вероятностей

stat_ztest, p_value_ztest = proportions_ztest(
    [m_a, m_b],
    [n_a, n_b],
    alternative='smaller' # так как H_1: p_a < p_b
)
print(f'Значение p: {p_value_ztest}')


if p_value_ztest > alpha:
    print(f'pvalue={p_value_ztest} > {alpha}')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'pvalue={p_value_ztest} < {alpha}')
    print('Нулевая гипотеза не находит подтверждения!')


n_a=4952, n_b=5011
m_a=1377, m_b=1480
p_a=0.27806946688206785, p_b=0.29535022949511075
Предпосылка о достаточном количестве данных выполняется!
Значение p: 0.028262547212292124
pvalue=0.028262547212292124 < 0.05
Нулевая гипотеза не находит подтверждения!


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

<div class="alert alert-block alert-info">
    Изначальная гипотеза:<br>
    упрощение интерфейса не приведет к повышению конверсии от регистрации в покупку.
    Альтернативная гипотеза:<br>
        упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации в системе конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта.<br>
    На основании анализа данных теста мы получили p-значение: 0.0282, что говорит нам о том, что нулевая гипотеза не подтвердилась. Мы также это проверили на количестве пользователей, которые перешли от регистрации к покупке в течении 7 дней с регистрации. Для теста использовались независимые группы пользователей и достаточное для теста количество сессий пользователей.<br>
    Ожидаемый эффект в изменении конверсии был почти достигнут, соответственно упрощение интерфейса было удачным решением для компании заказчика и привело к увеличению покупок пользователей.
</div>